In [1]:
# 1. Import Plotly
import plotly.graph_objects as go
import pandas as pd
import glob
from pathlib import Path

In [ ]:
# find relevant data files - use new timestamped format
csv_files = glob.glob("../logs/imu_log_*_mag.csv")
print(f"found {len(csv_files)} files: ")
for file in csv_files:
    print(f" - {file}")

# load the most recent file
if csv_files:
    latest_file = max(csv_files, key=lambda f: Path(f).stat().st_mtime)
    df = pd.read_csv(latest_file)
    print(f"\nLoaded {len(df)} data points from {Path(latest_file).name}")
else:
    print("No CSV files found. Run the logger and converter first:")
    print("  1. ./imu_logger")
    print("  2. python3 tools/log2csv.py data/logs/imu_log_*.bin")


found 1 files: 
 - ../logs/cal_data_mag.csv
Loaded 5512 data points


In [3]:
# check data structure
print("data shape:", df.shape);
print("\nColumns:", df.columns.tolist())
print("\nFirst few rows:")
df.head()

data shape: (5512, 5)

Columns: ['timestamp_ns', 'timestamp_s', 'x', 'y', 'z']

First few rows:


,timestamp_ns,timestamp_s,x,y,z
0,0,0.0,-0.05376,-0.01992,0.46600
1,0,0.0,-0.05616,-0.01936,0.46272
2,0,0.0,-0.05632,-0.01968,0.46440
3,0,0.0,-0.05752,-0.01520,0.46128
4,0,0.0,-0.05816,-0.02048,0.46712


In [ ]:
# compute time differences in milliseconds
df['time_diff_ms'] = df['timestamp_ns'].diff() / 1e6    
df['time_diff_ms'].fillna(0, inplace=True)

# display timestamp information
print(f"First timestamp: {df['timestamp_ns'].iloc[0]}")
print(f"First timestamp (s): {df['timestamp_s'].iloc[0]:.6f}")
print(f"Last timestamp (s): {df['timestamp_s'].iloc[-1]:.6f}")
print(f"Duration: {(df['timestamp_s'].iloc[-1] - df['timestamp_s'].iloc[0]):.3f} seconds")
print(f"\nAverage sample rate: {1000 / df['time_diff_ms'].mean():.1f} Hz")

# plot time jitter
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['time_diff_ms'], mode='lines+markers', name='Time Diff (ms)'))
fig.update_layout(title='Sample Time Differences', xaxis_title='Sample Index', yaxis_title='Time Difference (ms)')
fig.show()



First few rows with time_jitter:


In [5]:
# plot gyro data
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['x'], mode='lines+markers', name='Mag X'))
fig.add_trace(go.Scatter(x=df.index, y=df['y'], mode='lines+markers', name='Mag Y'))
fig.add_trace(go.Scatter(x=df.index, y=df['z'], mode='lines+markers', name='Mag Z'))
fig.update_layout(title='Magnetometer Data', xaxis_title='Sample Index', yaxis_title='Magnetic Field (µT)')
fig.show()

In [ ]:
# compute basic statistics for magnetometer data
mag_stats = {
    'x': {'mean': df['x'].mean(), 'std': df['x'].std(), 'min': df['x'].min(), 'max': df['x'].max()},
    'y': {'mean': df['y'].mean(), 'std': df['y'].std(), 'min': df['y'].min(), 'max': df['y'].max()},
    'z': {'mean': df['z'].mean(), 'std': df['z'].std(), 'min': df['z'].min(), 'max': df['z'].max()},
}   

print("\nMagnetometer Statistics (gauss):")
for axis, stats in mag_stats.items():
    print("-------------------------")
    print(f" {axis}: Mean = {stats['mean']:.4f}, Std = {stats['std']:.4f}")    
    print(f"     Range: [{stats['min']:.4f}, {stats['max']:.4f}]")

# compute magnitude
df['mag'] = (df['x']**2 + df['y']**2 + df['z']**2)**0.5
print(f"\nMagnitude: Mean = {df['mag'].mean():.4f}, Std = {df['mag'].std():.4f} gauss")


KeyError: 'gyro_x_dps'